# 01 OCI, RPO and AI Capacity Model

Key question: how much value should Oracle get for AI infrastructure
backlog when the same contracts require very large capex, financing and
execution through FY2030?

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from valuation import (
    DcfInputs,
    MarketPremiumInputs,
    OptionScenario,
    SegmentValuation,
    SotpInputs,
    build_sensitivity_grid,
    discounted_cash_flow,
    implied_margin_for_enterprise_value,
    implied_revenue_for_enterprise_value,
    market_premium_value,
    probability_weighted_option_value,
    sotp_equity_value,
)

DATA_DIR = next(
    candidate / "apps/notebooks/studies/oracle_valuation/data"
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "apps/notebooks/studies/oracle_valuation/data").exists()
)
pd.options.display.float_format = "{:,.2f}".format

In [ ]:
assumptions = pd.read_csv(DATA_DIR / "segment_assumptions.csv")
oci = (
    assumptions[assumptions["segment"].eq("OCI")]
    .pivot_table(index="case", columns="metric", values="value", aggfunc="first")
    .reindex(["bear", "base", "bull"])
)
oci

In [ ]:
oci_case = oci.assign(
    ebitda_usd_bn=lambda df: df["FY2030 revenue"] * df["EBITDA margin"],
    undiscounted_ev_usd_bn=lambda df: df["FY2030 revenue"]
    * df["EBITDA margin"]
    * df["EV/EBITDA multiple"],
    present_value_usd_bn=lambda df: df["undiscounted_ev_usd_bn"] * df["discount factor"],
)
oci_case

In [ ]:
revenue_margin_grid = build_sensitivity_grid(
    row_values=[95, 120, 145, 166, 190, 220],
    column_values=[0.25, 0.30, 0.34, 0.38, 0.42],
    formula=lambda revenue, margin: revenue * margin * 14.0 * 0.68,
    row_name="FY2030 OCI revenue (USD bn)",
    column_name="OCI EBITDA margin",
)
revenue_margin_grid

In [ ]:
rpo = assumptions[
    assumptions["segment"].eq("Market")
    & assumptions["metric"].eq("remaining performance obligations")
].iloc[0]["value"]
rpo_conversion_grid = build_sensitivity_grid(
    row_values=[0.30, 0.40, 0.50, 0.60],
    column_values=[0.25, 0.30, 0.35, 0.40],
    formula=lambda conversion, margin: rpo * conversion * margin * 12.0 * 0.68,
    row_name="RPO converted into steady-state OCI revenue",
    column_name="OCI EBITDA margin",
)
rpo_conversion_grid

In [ ]:
ax = oci_case[["FY2030 revenue", "ebitda_usd_bn", "present_value_usd_bn"]].plot(
    kind="bar", figsize=(9, 4), title="OCI FY2030 Case Outputs"
)
ax.set_ylabel("USD bn")
ax.set_xlabel("")
plt.tight_layout()